In [11]:
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import part1 # to use code from tutorial
import torch

# Shared Step: Create Datasets with Three Representations

In [12]:
# splitting data into train and test sets and folds for cross-validation

# get the dataframe
df = pd.read_pickle("./sampled_dataframe.pkl")

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["Party_orientation"])

# create cross-validation folds
my_seeds = [0] # we can have more
num_folds = 5

# function to create cross-validation folds
def create_cross_validation_sets(df, num_folds, seed):
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    folds = [df.iloc[idx] for idx in np.array_split(np.arange(len(df)), num_folds)]
    cross_validation_sets = []
    for i in range(num_folds):
        cv_test_df = folds[i]
        cv_train_df = pd.concat(folds[:i] + folds[i+1:])
        cross_validation_sets.append((cv_train_df, cv_test_df))
    return cross_validation_sets

my_cross_validation_sets = [create_cross_validation_sets(train_df, num_folds, seed) for seed in my_seeds]

In [13]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [14]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in set(x.split()):
            index = self.vocab.get(item)
            if index is not None:
                vector[index] = 1
        return vector

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column].str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))



In [15]:
label_names = sorted(df["Party_orientation"].unique())
column = "lemmas"   # "tokens" or "lemmas"
pairs = make_pairs(df, column, label_names)
vocab = get_vocab([text for text, _ in pairs], min_count=5)

In [16]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in set(x.split()):
            index = self.vocab.get(item)
            if index is not None:
                vector[index] += 1
        return vector

In [17]:
class Embedding(Encoder):
    def __init__(self, embeddings_map: pd.DataFrame):
        self.embeddings_map = embeddings_map
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.embeddings_map[self.embeddings_map['sentence'] == x].embeddings.squeeze()

In [18]:
embeddingDf = pd.read_pickle('./embeddings_sampled.pkl')

In [21]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(embeddingDf)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

Exp 1.B: Log-Lin with Bag of Words

Exp 1.C: Log-Lin with Sentence Embeddings

# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [20]:
my_classifiers["mlp_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

AttributeError: 'list' object has no attribute 'keys'

Exp 2.B: MLP with Bag of Words

In [ ]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

Exp 2.C: MLP with Sentence Embeddings

In [ ]:
my_classifiers["mlp_embeddings-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_embeddings-lemmas/seed-{seed}"
    my_classifiers["mlp_embeddings-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

# Experiment 3: CNN

Exp 3.A: CNN with Token Indicator

Exp 3.B: CNN with Bag of Words

Exp 3.C: CNN with Sentence Embeddings